# Linear models + regularization

Sparse truth, extra noise features. Compare OLS / Ridge / Lasso and sweep λ.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from interview_prep.metrics import mse, r2_score
from interview_prep.models import LinearRegression
from interview_prep.preprocessing import standardize
from interview_prep.regularizers import L1, L2, ElasticNet

rng = np.random.default_rng(0)
n, p = 120, 20
true_w = np.zeros(p)
true_w[:5] = [3.0, -2.0, 1.5, 0.0, 4.0]

X = rng.normal(size=(n, p))
y = X @ true_w + rng.normal(scale=0.5, size=n)
X_test = rng.normal(size=(400, p))
y_test = X_test @ true_w + rng.normal(scale=0.5, size=400)

X, mean, std = standardize(X)
X_test = (X_test - mean) / std


In [ ]:
def report(name, model):
    model.fit(X, y)
    print(
        f"{name:12} train MSE={mse(y, model.predict(X)):.3f}  "
        f"test MSE={mse(y_test, model.predict(X_test)):.3f}  "
        f"test R2={r2_score(y_test, model.predict(X_test)):.3f}  "
        f"||w||={np.linalg.norm(model.coef_):.3f}"
    )
    return model

ols = report("OLS", LinearRegression())
ridge = report("Ridge", LinearRegression(regularizer=L2(0.3)))
lasso = report("Lasso", LinearRegression(regularizer=L1(0.05), solver="gd", lr=0.05, max_iter=4000))
enet = report("ElasticNet", LinearRegression(regularizer=ElasticNet(0.2, l1_ratio=0.5), solver="gd", lr=0.05, max_iter=4000))


In [ ]:
lams = np.logspace(-3, 1, 16)
ridge_coefs, lasso_coefs = [], []
ridge_test, lasso_test = [], []

for lam in lams:
    r = LinearRegression(regularizer=L2(lam)).fit(X, y)
    l = LinearRegression(regularizer=L1(lam), solver="gd", lr=0.05, max_iter=3000).fit(X, y)
    ridge_coefs.append(r.coef_)
    lasso_coefs.append(l.coef_)
    ridge_test.append(mse(y_test, r.predict(X_test)))
    lasso_test.append(mse(y_test, l.predict(X_test)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(lams, ridge_coefs)
axes[0].set_xscale("log")
axes[0].set_title("Ridge coefficients")
axes[0].set_xlabel("λ")

axes[1].plot(lams, lasso_coefs)
axes[1].set_xscale("log")
axes[1].set_title("Lasso coefficients")
axes[1].set_xlabel("λ")

axes[2].plot(lams, ridge_test, label="Ridge test MSE")
axes[2].plot(lams, lasso_test, label="Lasso test MSE")
axes[2].set_xscale("log")
axes[2].set_xlabel("λ")
axes[2].legend()
fig.tight_layout()
plt.show()


Lasso should kill the noise coordinates first. Ridge shrinks everything and rarely hits exact zeros.
